# Phase 3 — Vocabulary Trimming

**เป้าหมาย:** ลบ dead tokens (ภาษาอื่นที่ไม่ใช้) จาก embedding ที่ tied (`embed_tokens` = `lm_head`)
**อ้างอิง:** Ushio et al. EMNLP 2023 (arXiv 2305.15020) — เหลือ vocab ~50% ยังรักษาคุณภาพได้
**เป้าตัวเลข:** vocab 128,256 → ~40,000 • embedding 394M → ~123M • รวม 3.21B → ~2.94B

**Input:** base model + `corpus_raw.txt` (จาก Phase 2)
**Output → Kaggle Dataset:** โมเดล vocab-trimmed + tokenizer ใหม่ + `id_mapping.json` + รายงานจำนวน token ที่ตัด

> Compute: CPU ล้วนพอ

In [ ]:
%pip install -q -U transformers sentencepiece
import os, json
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "scb10x/llama3.2-typhoon2-3b-instruct"
INPUT_DIR = "/kaggle/input"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CORPUS_FP = f"{INPUT_DIR}/<dataset>/corpus_raw.txt"   # TODO: ชี้ path จริง
MIN_FREQ = 1            # threshold ความถี่ขั้นต่ำที่จะเก็บ token
HF_TOKEN = os.environ.get("HF_TOKEN")

## 1. นับความถี่ทุก token บน corpus

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
freq = Counter()
with open(CORPUS_FP, encoding="utf-8") as f:
    for line in f:
        freq.update(tokenizer.encode(line, add_special_tokens=False))
print("token ที่ปรากฏ:", len(freq), "/", tokenizer.vocab_size)

## 2. เลือกเซต token ที่จะเก็บ
token ที่ผ่าน threshold + special tokens + ตัวเลข/เครื่องหมายพื้นฐาน

In [ ]:
keep = {tid for tid, c in freq.items() if c >= MIN_FREQ}
keep |= set(tokenizer.all_special_ids)
# เก็บตัวเลข/เครื่องหมายพื้นฐานเผื่อไว้
for ch in "0123456789.,:/-()%+ \n":
    keep.update(tokenizer.encode(ch, add_special_tokens=False))

keep_ids = sorted(keep)
print(f"เก็บ {len(keep_ids)} tokens (ตัดออก {tokenizer.vocab_size - len(keep_ids)})")

## 3. ตัดแถว embedding + สร้าง mapping old_id → new_id
tied embeddings → จัดการ `embed_tokens` ชุดเดียว (lm_head ผูกตาม)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, token=HF_TOKEN)
old2new = {old: new for new, old in enumerate(keep_ids)}

old_emb = model.get_input_embeddings().weight.data
new_emb = old_emb[torch.tensor(keep_ids)].clone()

model.resize_token_embeddings(len(keep_ids))
model.get_input_embeddings().weight.data.copy_(new_emb)
if model.get_output_embeddings() is not None and not model.config.tie_word_embeddings:
    model.get_output_embeddings().weight.data.copy_(new_emb)
model.config.vocab_size = len(keep_ids)
print("embedding ใหม่:", tuple(model.get_input_embeddings().weight.shape))

## 4. สร้าง tokenizer ใหม่ตาม mapping
> ⚠️ การ remap vocab ของ tokenizer (BPE/SentencePiece) ละเอียดอ่อน — ต้อง rebuild vocab+merges ให้ id ใหม่ต่อเนื่อง ตรวจ encode/decode ให้รอบคอบ

In [ ]:
# TODO: rebuild tokenizer ตาม keep_ids/old2new
#  - แนวทาง: แก้ไฟล์ tokenizer.json (vocab + merges) ให้เหลือเฉพาะ token ที่เก็บ แล้ว reindex
#  - หรือใช้ helper จากเปเปอร์ vocabtrimmer (arXiv 2305.15020)
raise NotImplementedError("rebuild tokenizer ตาม mapping")

## 5. Sanity test + นับ param + เซฟ

In [ ]:
for s in ["การยื่นขอทุนการศึกษา", "student dormitory registration"]:
    assert tokenizer.decode(tokenizer.encode(s, add_special_tokens=False)).strip(), s
print("✅ encode/decode ผ่าน")

total = sum(p.numel() for p in model.parameters())
print(f"param หลังตัด vocab: {total/1e9:.3f}B (เป้า ~2.94B)")

save_dir = os.path.join(OUT_DIR, "model_vocab_trimmed")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
json.dump({str(k): v for k, v in old2new.items()},
          open(os.path.join(OUT_DIR, "id_mapping.json"), "w"))
json.dump({"kept": len(keep_ids), "removed": tokenizer.vocab_size - len(keep_ids),
           "params_after": total},
          open(os.path.join(OUT_DIR, "vocab_trim_report.json"), "w"), indent=2)
print("เซฟที่:", save_dir, "→ push เป็น Kaggle Dataset สำหรับ Phase 4")